# Final slide figures

This notebook regenerates the vector PDF figures used by `tmp_slides/main.tex`. 
It reads the saved Group 5 pipeline artifacts and writes each figure to both 
`plots/final_slides/figures/` and `tmp_slides/figures/`.


In [ ]:

from pathlib import Path
import shutil
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle, FancyArrowPatch, FancyBboxPatch

ROOT = Path.cwd()
OUT_DIR = ROOT / "plots" / "final_slides" / "figures"
TEX_DIR = ROOT / "tmp_slides" / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
TEX_DIR.mkdir(parents=True, exist_ok=True)

ACORN_COLORS = {
    "ACORN-E": "#2f6f73",
    "ACORN-F": "#7b5f39",
    "ACORN-Q": "#8f3f46",
}
MODEL_LABELS = {
    "previous_day": "Previous day",
    "previous_week": "Previous week",
    "seasonal_mean": "Seasonal mean",
    "ridge": "Ridge",
    "xgboost": "XGBoost",
    "xgboost_by_acorn": "XGBoost by ACORN",
    "catboost": "CatBoost",
    "lightgbm": "LightGBM",
    "stack_regressor": "Stack regressor",
    "autogluon": "AutoGluon Tabular",
    "autogluon_timeseries": "AutoGluon TimeSeries",
}

mpl.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 160,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.size": 9,
    "axes.titlesize": 11,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linewidth": 0.6,
})


def save_fig(fig, filename):
    fig.patch.set_alpha(0)
    fig.patch.set_facecolor("none")
    for ax in fig.get_axes():
        ax.set_facecolor("none")
    for directory in (OUT_DIR, TEX_DIR):
        fig.savefig(
            directory / filename,
            bbox_inches="tight",
            format="pdf",
            transparent=True,
            facecolor="none",
            edgecolor="none",
        )
    plt.close(fig)


def add_panel_label(ax, label):
    ax.text(0.0, 1.04, label, transform=ax.transAxes, fontsize=10, fontweight="bold", va="bottom")

# Load artifacts
metrics = pd.read_csv(ROOT / "outputs/group5/metrics/validation_metrics.csv")
daily = pd.read_csv(ROOT / "outputs/group5/metrics/eda_daily_enriched.csv", parse_dates=["Date"])
profile = pd.read_csv(ROOT / "outputs/group5/metrics/eda_half_hour_profile.csv")
autocorr = pd.read_csv(ROOT / "outputs/group5/metrics/eda_daily_autocorrelation.csv")
val_daily = pd.read_csv(ROOT / "outputs/group5/metrics/validation_predictions_daily.csv", parse_dates=["timestamp"])
val_hh = pd.read_csv(ROOT / "outputs/group5/metrics/validation_predictions_half_hourly.csv", parse_dates=["timestamp"])
forecast_daily = pd.read_csv(ROOT / "outputs/group5/predictions/group_5_daily_predict.csv", parse_dates=["Date"])
forecast_hh = pd.read_csv(ROOT / "outputs/group5/predictions/group_5_half_hourly_predict.csv", parse_dates=["DateTime"])

# 01. Scope timeline
fig, ax = plt.subplots(figsize=(9.2, 3.4))
rows = [("Half-hourly", 1.0), ("Daily", 0.0)]
segments = {
    "Half-hourly": [
        (pd.Timestamp("2012-06-30 22:00"), pd.Timestamp("2013-12-16"), "Training history", "#d8d8d8"),
        (pd.Timestamp("2013-12-16"), pd.Timestamp("2014-01-12 23:30"), "Chronological validation", "#d49c5e"),
        (pd.Timestamp("2014-01-13"), pd.Timestamp("2014-01-14 23:30"), "48h forecast", "#4da4a9"),
    ],
    "Daily": [
        (pd.Timestamp("2012-07-01"), pd.Timestamp("2013-12-13"), "Training history", "#d8d8d8"),
        (pd.Timestamp("2013-12-13"), pd.Timestamp("2014-01-12"), "Chronological validation", "#d49c5e"),
        (pd.Timestamp("2014-01-13"), pd.Timestamp("2014-02-13"), "32-day forecast", "#c85a64"),
    ],
}
for name, y in rows:
    for start, end, label, color in segments[name]:
        x0 = mdates.date2num(start)
        width = mdates.date2num(end) - mdates.date2num(start)
        ax.add_patch(Rectangle((x0, y - 0.18), width, 0.36, facecolor=color, edgecolor="white", linewidth=1.2))
        mid = start + (end - start) / 2
        if label == "Training history":
            ax.text(mid, y, "Training history", ha="center", va="center", fontsize=8, color="#333333")
        elif label == "Chronological validation":
            ax.text(mid, y, "Validation", ha="center", va="center", fontsize=8, color="#333333")
        elif name == "Half-hourly":
            ax.annotate(
                "48 h\nforecast",
                xy=(mid, y),
                xytext=(pd.Timestamp("2014-01-25"), y + 0.34),
                ha="center",
                va="center",
                fontsize=8,
                color=color,
                arrowprops={"arrowstyle": "-", "color": color, "linewidth": 0.9, "shrinkA": 0, "shrinkB": 4},
            )
        else:
            ax.text(mid, y, "32 days\nforecast", ha="center", va="center", fontsize=8, color="white")
legend_handles = [
    Rectangle((0, 0), 1, 1, facecolor="#d8d8d8", edgecolor="white", label="Training history"),
    Rectangle((0, 0), 1, 1, facecolor="#d49c5e", edgecolor="white", label="Chronological validation"),
    Rectangle((0, 0), 1, 1, facecolor="#4da4a9", edgecolor="white", label="Final forecast window"),
]
ax.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, 1.12), ncol=3, frameon=False, fontsize=8)
ax.set_yticks([1, 0])
ax.set_yticklabels(["Half-hourly", "Daily"])
ax.set_xlim(pd.Timestamp("2012-06-15"), pd.Timestamp("2014-02-25"))
ax.set_ylim(-0.55, 1.65)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.set_xlabel("Calendar time")
ax.grid(axis="x", alpha=0.22)
ax.grid(axis="y", visible=False)
ax.text(pd.Timestamp("2014-01-13"), 1.42, "Forecast starts\n2014-01-13", ha="center", va="bottom", fontsize=7.5, color="#222")
ax.axvline(pd.Timestamp("2014-01-13"), color="#222", linewidth=0.8, linestyle="--")
save_fig(fig, "final_01_scope_timeline.pdf")

# 02. Daily trend and ACORN levels
fig, ax = plt.subplots(figsize=(8.8, 4.2))
for acorn, group in daily.sort_values("Date").groupby("Acorn"):
    rolled = group.set_index("Date")["Conso_kWh"].rolling(14, min_periods=1).mean()
    ax.plot(rolled.index, rolled.values, color=ACORN_COLORS[acorn], linewidth=1.8, label=acorn)
means = daily.groupby("Acorn")["Conso_kWh"].mean().sort_values(ascending=False)
for idx, (acorn, mean_val) in enumerate(means.items()):
    ax.text(0.02, 0.92 - idx * 0.08, f"{acorn}: {mean_val:.2f} kWh/day mean", transform=ax.transAxes,
            color=ACORN_COLORS[acorn], fontsize=8.5, fontweight="bold")
ax.set_title("Daily consumption has stable ranking and winter seasonality")
ax.set_ylabel("Daily consumption (kWh per client group)")
ax.set_xlabel("Date")
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.legend(loc="upper right", frameon=False)
save_fig(fig, "final_02_daily_trend.pdf")

# 03. Half-hour profile and weather relationship
fig, axes = plt.subplots(1, 2, figsize=(9.3, 3.8))
ax = axes[0]
profile = profile.copy()
profile["time_hour"] = profile["hour"] + profile["minute"] / 60
for acorn, group in profile.groupby("Acorn"):
    ax.plot(group["time_hour"], group["mean_conso_moy"], color=ACORN_COLORS[acorn], linewidth=2, label=acorn)
ax.set_title("Typical half-hourly profile")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Mean half-hourly consumption")
ax.set_xlim(0, 23.5)
ax.set_xticks([0, 6, 12, 18, 24])
ax.legend(frameon=False)
add_panel_label(ax, "A")

ax = axes[1]
for acorn, group in daily.groupby("Acorn"):
    g = group.dropna(subset=["temperatureMean", "Conso_kWh"]).copy()
    g["temp_bin"] = pd.cut(g["temperatureMean"], bins=np.arange(-2, 28, 2))
    binned = g.groupby("temp_bin", observed=True).agg(temp=("temperatureMean", "mean"), conso=("Conso_kWh", "mean"))
    ax.plot(binned["temp"], binned["conso"], marker="o", markersize=3.5, linewidth=1.8, color=ACORN_COLORS[acorn], label=acorn)
ax.set_title("Colder days consume more")
ax.set_xlabel("Daily mean temperature (C)")
ax.set_ylabel("Mean daily consumption (kWh)")
add_panel_label(ax, "B")
fig.tight_layout()
save_fig(fig, "final_03_profile_weather.pdf")

# 04. Autocorrelation and lag rationale
fig, ax = plt.subplots(figsize=(8.6, 4.0))
for acorn, group in autocorr.groupby("Acorn"):
    ax.plot(group["lag_days"], group["autocorrelation"], color=ACORN_COLORS[acorn], linewidth=1.9, label=acorn)
for lag, label in [(1, "yesterday"), (7, "same weekday"), (14, "two weeks")]:
    ax.axvline(lag, color="#222", linewidth=0.8, linestyle="--", alpha=0.55)
    ax.text(lag + 0.3, 0.18, label, rotation=90, va="bottom", fontsize=8, color="#333")
ax.set_title("Autocorrelation supports lag and rolling features")
ax.set_xlabel("Lag in days")
ax.set_ylabel("Daily autocorrelation")
ax.set_xlim(1, 30)
ax.set_ylim(0, 1.02)
ax.legend(frameon=False, loc="upper right")
save_fig(fig, "final_04_autocorrelation_lags.pdf")

# 05. Validation RMSE ranking
fig, axes = plt.subplots(1, 2, figsize=(10.2, 4.4))
for ax, freq, title, fmt in [(axes[0], "half_hourly", "Half-hourly RMSE", "{:.4f}"), (axes[1], "daily", "Daily RMSE", "{:.3f}")]:
    sub = metrics[(metrics["frequency"] == freq) & (metrics["acorn"] == "ALL")].copy()
    sub["label"] = sub["model"].map(MODEL_LABELS)
    sub = sub.sort_values("rmse", ascending=True)
    colors = ["#4da4a9" if i == 0 else "#c9c9c9" for i in range(len(sub))]
    ax.barh(sub["label"], sub["rmse"], color=colors, edgecolor="white", linewidth=0.8)
    ax.invert_yaxis()
    ax.set_title(title)
    ax.set_xlabel("RMSE")
    ax.grid(axis="x", alpha=0.25)
    ax.grid(axis="y", visible=False)
    xmax = sub["rmse"].max()
    for y, value in enumerate(sub["rmse"]):
        ax.text(value + xmax * 0.02, y, fmt.format(value), va="center", fontsize=7.5)
fig.tight_layout()
save_fig(fig, "final_05_validation_rmse.pdf")

# 06. Selected model validation traces
fig, axes = plt.subplots(2, 1, figsize=(9.0, 5.0), sharex=False)
sub = val_hh[val_hh["timestamp"] < val_hh["timestamp"].min() + pd.Timedelta(days=4)].copy()
agg = sub.groupby("timestamp")[["actual", "lightgbm"]].sum().reset_index()
axes[0].plot(agg["timestamp"], agg["actual"], color="#222222", linewidth=1.8, label="Actual")
axes[0].plot(agg["timestamp"], agg["lightgbm"], color="#4da4a9", linewidth=1.6, label="LightGBM")
axes[0].set_title("Half-hourly validation: first four days, all ACORNs summed")
axes[0].set_ylabel("Consumption")
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
axes[0].legend(frameon=False, ncol=2, loc="upper right")
add_panel_label(axes[0], "A")
agg = val_daily.groupby("timestamp")[["actual", "autogluon"]].sum().reset_index()
axes[1].plot(agg["timestamp"], agg["actual"], color="#222222", linewidth=1.8, label="Actual")
axes[1].plot(agg["timestamp"], agg["autogluon"], color="#c85a64", linewidth=1.6, label="AutoGluon Tabular")
axes[1].set_title("Daily validation: December holdout, all ACORNs summed")
axes[1].set_ylabel("kWh")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
axes[1].legend(frameon=False, ncol=2, loc="upper right")
add_panel_label(axes[1], "B")
fig.tight_layout()
save_fig(fig, "final_06_validation_traces.pdf")

# 07. Final forecasts
fig, axes = plt.subplots(1, 2, figsize=(10.0, 4.1))
ax = axes[0]
for acorn, group in forecast_hh.groupby("Acorn"):
    ax.plot(group["DateTime"], group["Conso_moy_predict"], color=ACORN_COLORS[acorn], linewidth=1.8, label=acorn)
ax.set_title("48-hour half-hourly forecast")
ax.set_xlabel("Forecast timestamp")
ax.set_ylabel("Predicted consumption")
ax.xaxis.set_major_formatter(mdates.DateFormatter("Jan %d\n%H:%M"))
ax.legend(frameon=False)
add_panel_label(ax, "A")

ax = axes[1]
for acorn, group in forecast_daily.groupby("Acorn"):
    ax.plot(group["Date"], group["Conso_kWh_predict"], color=ACORN_COLORS[acorn], linewidth=1.9, label=acorn)
ax.set_title("32-day daily forecast")
ax.set_xlabel("Forecast date")
ax.set_ylabel("Predicted kWh")
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.legend(frameon=False)
add_panel_label(ax, "B")
fig.tight_layout()
save_fig(fig, "final_07_final_forecasts.pdf")

# 08. Ensemble details
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.9))
weights = pd.DataFrame({
    "component": ["CatBoost L1", "LightGBMXT L1", "CatBoost L2", "RandomForest L2"],
    "weight": [50.0, 27.3, 18.2, 4.5],
})
axes[0].barh(weights["component"], weights["weight"], color=["#c85a64", "#4da4a9", "#d49c5e", "#8f8f8f"], edgecolor="white")
axes[0].invert_yaxis()
axes[0].set_xlabel("Weight in selected daily AutoGluon ensemble (%)")
axes[0].set_title("AutoGluon daily: WeightedEnsemble_L3_FULL")
axes[0].grid(axis="x", alpha=0.25)
axes[0].grid(axis="y", visible=False)
for y, value in enumerate(weights["weight"]):
    axes[0].text(value + 1.0, y, f"{value:.1f}%", va="center", fontsize=8)
add_panel_label(axes[0], "A")

ax = axes[1]
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
base_y = [0.78, 0.53, 0.28]
base_labels = ["Ridge", "XGBoost", "LightGBM"]
base_colors = ["#d8d8d8", "#d49c5e", "#4da4a9"]
for y, label, color in zip(base_y, base_labels, base_colors):
    box = FancyBboxPatch((0.05, y - 0.07), 0.28, 0.14, boxstyle="round,pad=0.02,rounding_size=0.03",
                         facecolor=color, edgecolor="white", linewidth=1.2)
    ax.add_patch(box)
    ax.text(0.19, y, label, ha="center", va="center", fontsize=9, fontweight="bold")
    ax.add_patch(FancyArrowPatch((0.34, y), (0.62, 0.53), arrowstyle="->", mutation_scale=12, linewidth=1.1, color="#333"))
meta = FancyBboxPatch((0.62, 0.41), 0.30, 0.24, boxstyle="round,pad=0.02,rounding_size=0.03",
                      facecolor="#f1f1f1", edgecolor="#333", linewidth=1.0)
ax.add_patch(meta)
ax.text(0.77, 0.53, "Ridge\nmeta-model", ha="center", va="center", fontsize=9, fontweight="bold")
ax.text(0.5, 0.08, "Stack regressor: a transparent ensemble baseline,\nnot the selected final model.", ha="center", va="center", fontsize=8)
ax.set_title("Stack regressor benchmark")
add_panel_label(ax, "B")
fig.tight_layout()
save_fig(fig, "final_08_ensemble_details.pdf")

# 09. Carbon estimate
GPU_POWER_KW = 0.300
CPU_MEMORY_OVERHEAD_KW = 0.150
PUE = 1.20
BULGARIA_CARBON_INTENSITY = 0.3653
scenarios = pd.DataFrame({"Scenario": ["8h lower", "12h expected", "18h time used"], "Runtime_h": [8, 12, 18]})
scenarios["Facility_kWh"] = scenarios["Runtime_h"] * (GPU_POWER_KW + CPU_MEMORY_OVERHEAD_KW) * PUE
scenarios["CO2e_kg"] = scenarios["Facility_kWh"] * BULGARIA_CARBON_INTENSITY
fig, ax = plt.subplots(figsize=(7.2, 3.8))
bars = ax.bar(scenarios["Scenario"], scenarios["CO2e_kg"], color=["#4da4a9", "#d49c5e", "#c85a64"], edgecolor="white")
for bar, value in zip(bars, scenarios["CO2e_kg"]):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.05, f"{value:.2f} kg", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.set_title("Estimated CO2e for the all-in AutoGluon calibration")
ax.set_ylabel("kg CO2e")
ax.set_xlabel("Runtime scenario")
ax.set_ylim(0, scenarios["CO2e_kg"].max() * 1.25)
ax.text(0.02, 0.92, "Formula: runtime x 0.45 kW IT power x PUE 1.20 x 365.3 gCO2e/kWh", transform=ax.transAxes, fontsize=8)
ax.grid(axis="y", alpha=0.25)
ax.grid(axis="x", visible=False)
save_fig(fig, "final_09_carbon_estimate.pdf")

print("Generated slide figures:")
for path in sorted(OUT_DIR.glob("final_*.pdf")):
    print("-", path.relative_to(ROOT))
